In [ ]:
# # LightOnOCR-2-1B — OCR Tool
#
# **LightOnOCR-2-1B** là model OCR end-to-end 1B tham số của [LightOn](https://lighton.ai),
# được huấn luyện với RLVR để đạt độ chính xác tối đa.
#
# ### Highlights
# - ⚡ **Tốc độ**: 3.3× nhanh hơn Chandra OCR, 1.7× nhanh hơn OlmOCR
# - 💸 **Hiệu quả**: 5.71 trang/giây trên H100 (~493k trang/ngày), <$0.01/1000 trang
# - 🧾 **Đa dạng**: Bảng biểu, hoá đơn, form, bố cục nhiều cột, ký hiệu toán học
# - 🌍 **Đa ngôn ngữ**: en, fr, de, es, it, nl, pt, sv, da, zh, ja
# - 📄 **Paper**: https://arxiv.org/pdf/2601.14251
# - 🚀 **Demo**: https://huggingface.co/spaces/lightonai/LightOnOCR-2-1B-Demo
#
# ### Model Variants
# | Variant | Mô tả |
# |---------|--------|
# | **LightOnOCR-2-1B** | Model OCR tốt nhất ✅ (đang dùng) |
# | LightOnOCR-2-1B-base | Base model, lý tưởng để fine-tune |
# | LightOnOCR-2-1B-bbox | Kèm dự đoán bounding box cho ảnh nhúng |
# | LightOnOCR-2-1B-bbox-base | Base bbox model để fine-tune |
# | LightOnOCR-2-1B-ocr-soup | Merged variant, thêm robust |
# | LightOnOCR-2-1B-bbox-soup | OCR + bbox kết hợp |
#
# ### Lưu ý cài đặt
# ```bash
# pip install "transformers>=5.0.0" torch pillow pypdfium2 accelerate
# ```
# - Model đã tải về: `LightOnOCR-2-1B/` (local)
# - Để tải từ HuggingFace: thay `MODEL_DIR = "lightonai/LightOnOCR-2-1B"`
# - Render PDF tốt nhất ở **200 DPI**, kích thước ảnh dài nhất **1540px**


In [1]:
# Yêu cầu: Python >= 3.10, transformers >= 5.0.0
# pip install "transformers>=5.0.0" torch pillow pypdfium2
import torch
import gc
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor
from PIL import Image
import pypdfium2 as pdfium
import os

# --- CẤU HÌNH ---
MODEL_DIR = "LightOnOCR-2-1B"

# MPS: float32, CUDA: bfloat16
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32 if DEVICE == "mps" else torch.bfloat16

# Giảm IMG_MAX để tránh OOM crash trên MPS
PDF_DPI  = 150        # Giảm từ 200 → 150 để render ảnh nhỏ hơn
IMG_MAX  = 1024       # Giảm từ 1540 → 1024 để giảm áp lực GPU RAM
MAX_TOKENS = 1024     # Giảm từ 2048 → 1024

print(f"Device: {DEVICE} | dtype: {DTYPE}")
print(f"PDF_DPI={PDF_DPI}, IMG_MAX={IMG_MAX}, MAX_TOKENS={MAX_TOKENS}")


/Users/wtechkane/Workspace/Projects/ocr-python/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps | dtype: torch.float32
PDF_DPI=150, IMG_MAX=1024, MAX_TOKENS=1024


In [2]:
# --- NẠP MODEL (chạy 1 lần) ---
print("Đang nạp model...")

processor = LightOnOcrProcessor.from_pretrained(MODEL_DIR, trust_remote_code=True)

model = LightOnOcrForConditionalGeneration.from_pretrained(
    MODEL_DIR,
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE)
model.eval()

print("✅ Nạp model thành công!")


Using `use_fast=True` but `torchvision` is not available. Falling back to the slow image processor.


Đang nạp model...


You are using a model of type `mistral3` to instantiate a model of type `lighton_ocr`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 100%|██████████| 532/532 [00:02<00:00, 208.15it/s]


✅ Nạp model thành công!


In [3]:
# --- HÀM TIỆN ÍCH ---

def free_memory():
    """Giải phóng bộ nhớ GPU/MPS."""
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "cpu":
        pass  # Không cần làm gì đặc biệt cho CPU


def resize_if_needed(img: Image.Image) -> Image.Image:
    """Giữ aspect ratio, giới hạn cạnh dài nhất = IMG_MAX."""
    w, h = img.size
    longest = max(w, h)
    if longest > IMG_MAX:
        scale = IMG_MAX / longest
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img


def ocr_image(img: Image.Image) -> str:
    """OCR một ảnh PIL → Markdown. An toàn với MPS."""
    img = resize_if_needed(img.convert("RGB"))

    conversation = [
        {
            "role": "user",
            "content": [{"type": "image", "url": img}],
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        # Chuyển tensor sang đúng device/dtype
        inputs = {
            k: v.to(device=DEVICE, dtype=DTYPE) if v.is_floating_point() else v.to(DEVICE)
            for k, v in inputs.items()
        }

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_TOKENS,
                do_sample=False,   # Tắt sampling để ổn định hơn trên MPS
            )

        input_len = inputs["input_ids"].shape[1]
        generated_ids = output_ids[0, input_len:]
        result = processor.decode(generated_ids, skip_special_tokens=True)

        # Xoá tensors khỏi bộ nhớ ngay sau khi dùng xong
        del inputs, output_ids, generated_ids
        free_memory()

        return result

    except Exception as e:
        free_memory()
        return f"❌ Lỗi OCR: {e}"


def pdf_to_images(pdf_path: str) -> list[Image.Image]:
    """Render tất cả trang PDF → list PIL Image."""
    doc = pdfium.PdfDocument(pdf_path)
    images = []
    for page in doc:
        bitmap = page.render(scale=PDF_DPI / 72, rotation=0)
        pil_img = bitmap.to_pil()
        images.append(pil_img)
    doc.close()
    return images


def process_file(file_path: str, output_path: str | None = None) -> str:
    """Convert PDF hoặc ảnh → Markdown. Ghi từng trang ra file ngay lập tức."""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        print(f"📄 PDF: {file_path}")
        images = pdf_to_images(file_path)
        pages  = []

        # Mở file output sẵn để ghi streaming (tránh mất kết quả nếu crash cuối)
        f_out = None
        if output_path:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            f_out = open(output_path, "w", encoding="utf-8")

        try:
            for i, img in enumerate(images):
                print(f"   Trang {i+1}/{len(images)} ({img.size[0]}×{img.size[1]})...", end=" ", flush=True)
                page_md = f"<!-- Trang {i+1} -->\n{ocr_image(img)}"
                pages.append(page_md)
                print("✓")

                # Ghi ngay vào file để không mất dữ liệu nếu crash sau
                if f_out:
                    if i > 0:
                        f_out.write("\n\n---\n\n")
                    f_out.write(page_md)
                    f_out.flush()

                # Xoá ảnh khỏi RAM ngay sau khi xử lý xong
                del img
                images[i] = None  # type: ignore
                free_memory()
        finally:
            if f_out:
                f_out.close()

        return "\n\n---\n\n".join(pages)

    elif ext in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}:
        print(f"🖼️  Ảnh: {file_path}")
        result = ocr_image(Image.open(file_path).convert("RGB"))
        if output_path:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(result)
        return result

    else:
        return f"❌ Định dạng không hỗ trợ: {ext}"


In [4]:
# --- CHẠY ---
input_path  = "input/scanned3.pdf"  # ← đổi thành file của bạn
output_path = "output/scanned3.md"

if os.path.exists(input_path):
    result = process_file(input_path, output_path=output_path)
    print(f"\n✅ Xong! → {output_path}")
else:
    print(f"❌ Không tìm thấy: {input_path}")

📄 PDF: input/scanned3.pdf
   Trang 1/7 (1242×1755)... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✓
   Trang 2/7 (1242×1755)... ✓
   Trang 3/7 (1242×1755)... ✓
   Trang 4/7 (1242×1755)... ✓
   Trang 5/7 (1242×1755)... ✓
   Trang 6/7 (1242×1755)... ✓
   Trang 7/7 (1242×1755)... ✓

✅ Xong! → output/scanned3.md


In [30]:
# --- TEST OCR ẢNH ---
# Đặt đường dẫn ảnh cần test vào đây
TEST_IMAGE = "input/orientation_and_cropping/image_test_4.png"  # ← đổi thành file ảnh của bạn
# Lưu kết quả ra file (tuỳ chọn)
out_txt = "output/orientation_and_cropping/image_test_4.md"

if os.path.exists(TEST_IMAGE):
    print(f"🖼️  Đang OCR: {TEST_IMAGE}")
    img = Image.open(TEST_IMAGE).convert("RGB")
    print(f"   Kích thước gốc : {img.size[0]}×{img.size[1]}")

    img_resized = resize_if_needed(img)
    print(f"   Kích thước sau resize: {img_resized.size[0]}×{img_resized.size[1]}")

    result = ocr_image(img)


    with open(out_txt, "w", encoding="utf-8") as f:
        f.write(result)
    print(f"\n✅ Đã lưu kết quả → {out_txt}")
else:
    print(f"❌ Không tìm thấy ảnh: {TEST_IMAGE}")

🖼️  Đang OCR: input/orientation_and_cropping/image_test_4.png
   Kích thước gốc : 1133×714
   Kích thước sau resize: 1024×645

✅ Đã lưu kết quả → output/orientation_and_cropping/image_test_4.md
